# BLaVe-CoT — Demo LoRA BLIP-2 (5 epoch trên Colab T4)

Notebook chạy demo nhỏ (~118 mẫu, **5 epoch**, LoRA fp16) trên GPU T4 Colab.

## Cách dùng
1. Mở notebook này trong VS Code (đã cài extension **Google Colab**) hoặc upload trực tiếp lên https://colab.research.google.com.
2. Kernel selector (góc phải trên) → **Connect to a Colab runtime** → **T4 GPU** (KHÔNG chọn CPU/None).
3. Trước khi chạy: đảm bảo `demo_pack.zip` đã ở `MyDrive/blave_train/demo_pack.zip` trong Google Drive của bạn.
4. Chạy lần lượt các cell bên dưới (hoặc bấm **Run All**).

**Lưu ý:** T4 16GB đủ chạy fp16 → notebook tự tắt QLoRA + upgrade `torchao` để tránh ImportError đã gặp.

## 1. Kiểm tra GPU + clone repo

Output phải hiện `CUDA: True | Tesla T4`. Nếu hiện `CPU` → dừng lại, đổi runtime sang T4 GPU rồi chạy lại.

In [ ]:
import torch, os
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Không có GPU! Runtime → Change runtime type → T4 GPU rồi chạy lại.'

%cd /content
if not os.path.isdir('blave_train'):
    !git clone https://github.com/HuynhKhoaIT/blave_train.git
%cd blave_train
!git checkout -- config.py 2>/dev/null    # bỏ thay đổi local (do sed) để git pull không xung đột
!git pull
!ls *.py

## 2. Cài lib + fix torchao + lấy `demo_pack.zip` từ Google Drive

Yêu cầu: `demo_pack.zip` đã có sẵn ở `MyDrive/blave_train/demo_pack.zip` (tạo folder `blave_train` trong MyDrive nếu chưa có).

Cell này tự upgrade `torchao` lên ≥0.16.0 để khớp với `peft` mới (Colab mặc định có 0.10.0 → gây ImportError).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U "torchao>=0.16.0"

from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/blave_train/demo_pack.zip .
!unzip -q -o demo_pack.zip
!ls data/ && ls data/vizwiz/train | head -3

## 3. Override config: 5 epoch + tắt QLoRA

- Repo GitHub đang để `num_epochs=2` (bản demo) → override thành 5.
- Repo GitHub đang để `use_qlora=True` → tắt thành False vì T4 16GB đủ chạy fp16, tránh phụ thuộc bitsandbytes/torchao.

In [ ]:
!sed -i 's/"num_epochs": 2,/"num_epochs": 5,/' config.py
!sed -i 's/"use_qlora": True/"use_qlora": False/' config.py
!python config.py | grep -E 'num_epochs|use_qlora'

## 4. Xoá checkpoint cũ + train 5 epoch

Lần đầu sẽ tải BLIP-2 base (~15GB) từ HuggingFace — mất ~3 phút. Sau đó train 5 epoch trên 118 mẫu LoRA fp16 → khoảng 8-10 phút trên T4.

In [ ]:
!rm -rf ./blip2_lora_demo
!python train.py

## 5. Inference — so sánh trước/sau fine-tune

In [ ]:
import json
sample = json.load(open('data/train_converted.json'))[0]
img = f"data/vizwiz/train/{sample['image']}"
print(f"Question: {sample['question']}")
print(f"Ground truth: {sample['answer']}\n")
!python infer.py --adapter ./blip2_lora_demo/final --image "{img}" --question "{sample['question']}" --compare --candidates 3

## 6. Đánh giá trên nhiều mẫu

Chạy 5 mẫu đầu để xem adapter có khác BLIP-2 gốc ở mẫu khó không (1 mẫu không đủ kết luận).

In [ ]:
import json
samples = json.load(open('data/train_converted.json'))[:5]
for i, s in enumerate(samples, 1):
    img = f"data/vizwiz/train/{s['image']}"
    print(f"\n=== Mẫu {i} ===")
    print(f"Q : {s['question']}")
    print(f"GT: {s['answer']}")
    !python infer.py --adapter ./blip2_lora_demo/final --image "{img}" --question "{s['question']}" --compare --candidates 3